# Week 1 & 2: Exploratory Data Analysis & Text Preprocessing
### Project: Intelligent Customer Review Sentiment Analysis
This notebook covers:
1. Dataset loading (Amazon Product Reviews)
2. 3-Class Rating Mapping (1-2 Stars = Negative, 3 Stars = Neutral, 4-5 Stars = Positive)
3. Class balance and review length distribution
4. NLP Preprocessing Pipeline: HTML cleaning, contractions, tokenization, stopword preservation
5. TF-IDF feature extraction & Baseline Logistic Regression / Naive Bayes classifier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Set visual style
sns.set_theme(style='whitegrid')
print('Libraries imported successfully!')

## 1. Load Dataset and Map Ratings to 3-Class Sentiments

In [ ]:
# Load reviews
df = pd.read_csv('../backend/data/sample_reviews.csv')
print(f'Total reviews loaded: {len(df)}')
display(df.head())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Sentiment distribution
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='sentiment', palette='viridis', order=['Negative', 'Neutral', 'Positive'])
plt.title('Sentiment Class Distribution')
plt.xlabel('Sentiment Class')
plt.ylabel('Count')
plt.show()

# Review length distribution
df['char_length'] = df['review_text'].apply(len)
df['word_count'] = df['review_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 4))
sns.histplot(data=df, x='word_count', hue='sentiment', kde=True, bins=15, palette='Set2')
plt.title('Review Word Count by Sentiment')
plt.xlabel('Words per Review')
plt.show()

## 3. Text Preprocessing Implementation
Preserving negations (e.g. *not*, *never*, *cannot*) is critical for sentiment classification accuracy.

In [ ]:
from app.services.preprocessor import TextPreprocessor

preprocessor = TextPreprocessor()
df['cleaned_text'] = df['review_text'].apply(lambda x: preprocessor.clean_text(x, remove_stopwords=True))

print('Sample Raw vs Cleaned Text:')
for i in range(3):
    print(f'RAW:     {df["review_text"].iloc[i]}')
    print(f'CLEANED: {df["cleaned_text"].iloc[i]}\n')

## 4. Baseline Machine Learning Model (TF-IDF + Logistic Regression)

In [ ]:
X = df['cleaned_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

baseline_clf = LogisticRegression(max_iter=1000)
baseline_clf.fit(X_train_vec, y_train)

y_pred = baseline_clf.predict(X_test_vec)
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Negative', 'Neutral', 'Positive'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
plt.title('Baseline TF-IDF Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()